Exploration notebook. The shipped `models/checkpoint.pt` is produced by
`python -m src.train` (search, then a refit on all 60,000 images); see DEVELOPMENT.md.

In [41]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import torch


if torch.backends.mps.is_available():
    print("M4 GPU is available via MPS! 🚀")
else:
    print("Defaulting to CPU. Ensure you installed the Mac-native PyTorch.")


M4 GPU is available via MPS! 🚀


In [2]:
# 1. Set the device dynamically
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: mps


In [ ]:
import os, sys
from torch import nn
from torch.utils.data import DataLoader
import torch.optim as optim
sys.path.append("..")
from src import data as mnist_data, nn_model, preprocessing

# Training runs re-seed from `seed` before building each model and shuffle order.
seed = 11
# Isolate split randomness from model initialization and other notebook work.
split_seed = 11

# Raw training images (N, 1, 28, 28) uint8 and labels; downloads if missing.
images, labels = mnist_data.load_mnist_train("../data")

# train/validation split
train_indices, validation_indices = mnist_data.split_indices(
    len(labels), 0.8, generator=torch.Generator().manual_seed(split_seed)
)

In [ ]:
# train/validation data preprocessing
train_images, train_labels = mnist_data.select(images, labels, train_indices)
validation_images, validation_labels = mnist_data.select(
    images, labels, validation_indices
)

# Statistics come from the training subset only and are reused for validation.
train_u, train_std = mnist_data.fit_statistics(train_images)
training_prepared = mnist_data.to_dataset(train_images, train_labels, train_u, train_std)
validation_prepared = mnist_data.to_dataset(
    validation_images, validation_labels, train_u, train_std
)

In [ ]:
# hyperparameters
batch_sizes = [32, 64, 128]
learning_rates = [1e-4, 1e-3, 1e-2]
weight_decay = [0, 1e-4, 1e-3, 1e-2]

In [45]:
import pandas as pd
result = []

In [ ]:
for batch_size in batch_sizes:
    for learning_rate in learning_rates:
        for decay in weight_decay:
            print(f"Training with batch size: {batch_size}, learning rate: {learning_rate}, weight decay: {decay}")
            # Re-seed per configuration: every candidate gets the same initial
            # weights and shuffle order, independent of its position in the loop.
            torch.manual_seed(seed)
            shuffle_generator = torch.Generator().manual_seed(seed)

            model = nn_model.NeuralNetwork().to(device)
            optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=decay)
            loss_function = nn.CrossEntropyLoss()

            train_dataloader = DataLoader(training_prepared, batch_size=batch_size, generator=shuffle_generator, shuffle=True)
            validation_dataloader = DataLoader(validation_prepared, batch_size=batch_size, shuffle=False)

            train_losses = []
            eval_losses = []

            for epoch in range(5):  # Train for 5 epochs
                mean_train_loss, train_size, batch_train_losses = nn_model.train_one_epoch(
                    model, optimizer, loss_function, train_dataloader
                )
                mean_eval_loss, eval_size, batch_eval_losses = nn_model.model_eval(
                    model, validation_dataloader, loss_function
                )

                train_losses.extend(batch_train_losses)
                eval_losses.extend(batch_eval_losses)

            result.append({ "seed": seed, "batch_size": batch_size, "learning_rate": learning_rate, "weight_decay": decay, "train_loss": mean_train_loss, "eval_loss": mean_eval_loss }) 

In [47]:
result = pd.DataFrame(result)

In [50]:
idx = result['eval_loss'].idxmin()
result.iloc[idx]

batch_size       64.000000
learning_rate     0.001000
weight_decay      0.000000
train_loss        0.030620
eval_loss         0.036326
Name: 24, dtype: float64

# model training with chosen hyperparameters

In [51]:
train_std, train_u

(tensor(70.2115), tensor(25.5334))

In [ ]:
batch_size = 64
learning_rate = 1e-3
weight_decay = 0
epochs = 5
model_result = []
# Same seeding as the search loop, so this run should reproduce the chosen
# configuration's search result.
torch.manual_seed(seed)
shuffle_generator = torch.Generator().manual_seed(seed)

final_model = nn_model.NeuralNetwork().to(device)
optimizer = optim.Adam(final_model.parameters(), lr=learning_rate, weight_decay=weight_decay)
loss_function = nn.CrossEntropyLoss()

train_dataloader = DataLoader(training_prepared, batch_size=batch_size, generator=shuffle_generator, shuffle=True)
validation_dataloader = DataLoader(validation_prepared, batch_size=batch_size, shuffle=False)

train_losses = []
eval_losses = []

for epoch in range(epochs):
    mean_train_loss, train_size, batch_train_losses = nn_model.train_one_epoch(
        final_model, optimizer, loss_function, train_dataloader
    )
    mean_eval_loss, eval_size, batch_eval_losses = nn_model.model_eval(
        final_model, validation_dataloader, loss_function
    )

    train_losses.extend(batch_train_losses)
    eval_losses.extend(batch_eval_losses)

model_result.append({ "seed": seed, "batch_size": batch_size, "learning_rate": learning_rate, "weight_decay": weight_decay, "train_loss": mean_train_loss, "eval_loss": mean_eval_loss }) 

In [59]:
model_result

[{'batch_size': 64,
  'learning_rate': 0.001,
  'weight_decay': 0,
  'train_loss': 0.03349075822741725,
  'eval_loss': 0.042647618104082845}]

In [ ]:
from pathlib import Path

# Measure validation accuracy for this candidate.
final_model.eval()
correct = 0
samples = 0
with torch.no_grad():
    for batch_images, batch_labels in validation_dataloader:
        batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)
        predictions = final_model(batch_images).argmax(dim=1)
        correct += (predictions == batch_labels).sum().item()
        samples += batch_labels.size(0)
validation_accuracy = correct / samples

checkpoint = {
    'architecture': 'lenet5_relu_maxpool_dense_v1',
    'model_state_dict': final_model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epochs_completed': epochs,
    'training_seed': seed,
    'split_seed': split_seed,
    'train_samples': len(training_prepared),
    'validation_samples': len(validation_prepared),
    'hyperparameters': {
        'optimizer': 'Adam',
        'batch_size': batch_size,
        'learning_rate': learning_rate,
        'weight_decay': weight_decay,
    },
    'preprocessing': {
        'input_shape': [1, 28, 28],
        'input_dtype': 'uint8',
        'pixel_range': [0, 255],
        'padding_per_side': 2,
        'padding_value': 0,
        'output_dtype': 'float32',
        'scale_to_unit_interval': False,
        'train_mean': train_u,
        'train_std': train_std,
    },
    'validation_loss': mean_eval_loss,
    'validation_accuracy': validation_accuracy,
}
# Exploration copy only; the shipped checkpoint comes from `python -m src.train`.
checkpoint_path = Path("../runs/notebook/checkpoint.pt")
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(checkpoint, checkpoint_path)

# Reconstruct a fresh model; no optimizer is needed for inference.
loaded_checkpoint = torch.load(
    checkpoint_path, map_location=device, weights_only=True
)
loaded_model = nn_model.NeuralNetwork().to(device)
loaded_model.load_state_dict(loaded_checkpoint['model_state_dict'])
loaded_model.eval()

# Rebuild inputs from raw validation images using saved preprocessing.
raw_images = images[validation_indices[:64]]
padded_images = preprocessing.padding_image(raw_images)
saved_preprocessing = loaded_checkpoint['preprocessing']

original_inputs = preprocessing.normalize_image(
    padded_images, train_u, train_std
).to(device)
loaded_inputs = preprocessing.normalize_image(
    padded_images,
    saved_preprocessing['train_mean'],
    saved_preprocessing['train_std'],
).to(device)

with torch.no_grad():
    original_logits = final_model(original_inputs)
    loaded_logits = loaded_model(loaded_inputs)

torch.testing.assert_close(
    loaded_logits, original_logits, rtol=1e-5, atol=1e-6
)
print(f"Validation accuracy: {validation_accuracy:.2%}")
print("Checkpoint reload verified: logits match.")
